In [6]:
import os
import pyproj

# Fix PROJ database conflict caused by PostgreSQL / PostGIS installation
os.environ["PROJ_LIB"] = pyproj.datadir.get_data_dir()
os.environ["PROJ_DATA"] = pyproj.datadir.get_data_dir()

import numpy as np
import pandas as pd
import planetary_computer
import pystac_client
import rioxarray
from pyproj import Transformer

# 1. Connect to Microsoft Planetary Computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


def get_green_coverage(
    id: int, lat: float, lon: float, buffer_meters: int = 1000
) -> float:
    """Calculates percentage of green coverage (Trees=10, Grass=30)

    within a buffer radius (in meters) around a Lat/Lon point.
    """
    buffer_deg = buffer_meters / 111_000.0
    bbox = [
        lon - buffer_deg,
        lat - buffer_deg,
        lon + buffer_deg,
        lat + buffer_deg,
    ]

    # Query the STAC API (using updated items() method)
    search = catalog.search(collections=["esa-worldcover"], bbox=bbox)
    items = list(search.items())

    if not items:
        return 0.0

    item = items[0]
    map_asset_url = item.assets["map"].href

    # Stream only the required bounding box slice
    rds = rioxarray.open_rasterio(map_asset_url, masked=True)
    rds_clipped = rds.rio.clip_box(*bbox)

    # Reproject raster to UTM 36N (Egypt projection in meters)
    rds_projected = rds_clipped.rio.reproject("EPSG:32636")

    # Transform center point to UTM 36N
    transformer = Transformer.from_crs(
        "EPSG:4326", "EPSG:32636", always_xy=True
    )
    point_utm_x, point_utm_y = transformer.transform(lon, lat)

    # Calculate distance grid
    x_coords, y_coords = rds_projected.x.values, rds_projected.y.values
    grid_x, grid_y = np.meshgrid(x_coords, y_coords)
    dist_from_center = (
        (grid_x - point_utm_x) ** 2 + (grid_y - point_utm_y) ** 2
    ) ** 0.5

    # Filter pixels inside circular buffer
    circle_mask = dist_from_center <= buffer_meters
    raster_data = rds_projected.values[0]
    valid_pixels = raster_data[circle_mask]

    if len(valid_pixels) == 0:
        return 0.0

    # Count Trees (10) and Grassland (30)
    green_pixels = (valid_pixels == 10) | (valid_pixels == 30)
    green_percentage = (green_pixels.sum() / len(valid_pixels)) * 100

    print(f"Done {id}")

    return round(float(green_percentage), 2)

In [2]:
df = pd.read_csv("../data/locations.csv")
df.head(5)

,id,lat,lon
0,1,30.222,31.732
1,2,30.238,31.700
2,3,29.853,31.387
3,4,29.985,30.963
4,5,29.941,30.908


In [7]:
print("Calculating green coverage for Egypt dataset...")

df["green_coverage_pct"] = df.apply(
    lambda row: get_green_coverage(
        row['id'], row["lat"], row["lon"], buffer_meters=1000
    ),
    axis=1,
)

Calculating green coverage for Egypt dataset...
Done 1.0
Done 2.0
Done 3.0
Done 4.0
Done 5.0
Done 6.0
Done 7.0
Done 8.0
Done 9.0
Done 10.0
Done 11.0
Done 12.0
Done 13.0
Done 14.0
Done 15.0
Done 16.0
Done 17.0
Done 18.0
Done 19.0
Done 20.0
Done 21.0
Done 22.0
Done 23.0
Done 24.0
Done 25.0
Done 26.0
Done 27.0
Done 28.0
Done 29.0
Done 30.0
Done 31.0
Done 32.0
Done 33.0
Done 34.0
Done 35.0
Done 36.0
Done 37.0
Done 38.0
Done 39.0
Done 40.0
Done 41.0
Done 42.0
Done 43.0
Done 44.0
Done 45.0
Done 46.0
Done 47.0
Done 48.0
Done 49.0
Done 50.0
Done 51.0
Done 52.0
Done 53.0
Done 54.0
Done 55.0
Done 56.0
Done 57.0
Done 58.0
Done 59.0
Done 60.0
Done 61.0
Done 62.0
Done 63.0
Done 64.0
Done 65.0
Done 66.0
Done 67.0
Done 68.0
Done 69.0
Done 70.0
Done 71.0
Done 72.0
Done 73.0
Done 74.0
Done 75.0
Done 76.0
Done 77.0
Done 78.0
Done 79.0
Done 80.0
Done 81.0
Done 82.0
Done 83.0
Done 84.0
Done 85.0
Done 86.0
Done 87.0
Done 88.0
Done 89.0
Done 90.0
Done 91.0
Done 92.0
Done 93.0
Done 94.0
Done 95.0
Done 96.0
D

In [8]:
df.head(5)

,id,lat,lon,green_coverage_pct
0,1,30.222,31.732,1.79
1,2,30.238,31.700,0.10
2,3,29.853,31.387,1.05
3,4,29.985,30.963,4.21
4,5,29.941,30.908,0.30


In [9]:
df['green_coverage_pct'].max()

np.float64(77.06)

In [10]:
df.to_csv("../data/greenCovarage.csv", sep=",", index=False, header=True)